# Project 68 — Local Cost & Latency Benchmark

**Compare speed and output quality across local Ollama models.**

| Component | Choice |
|-----------|--------|
| LLM | Ollama (multiple models) |
| Analysis | pandas |
| Interface | Jupyter |

## 1 · What You Will Learn

1. Benchmark **multiple local models** on the same tasks
2. Measure **latency, throughput, words/second**
3. Compare **output quality vs speed** tradeoffs
4. Choose the **right model** for each use case

## 2 · Why This Project Matters

Not all tasks need the biggest model. Find the sweet spot between quality and speed.

## 3 · Setup

In [ ]:
# !pip install -q langchain langchain-ollama pandas

## 4 · Imports

In [ ]:
import json, time, subprocess
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODELS = ['qwen3:8b']
try:
    r = subprocess.run(['ollama', 'list'], capture_output=True, text=True, timeout=5)
    avail = [l.split()[0] for l in r.stdout.strip().split('\n')[1:] if l.strip()]
    for c in ['qwen3:4b', 'llama3.2:3b', 'phi4-mini', 'gemma3:4b']:
        if c in avail and c not in MODELS: MODELS.append(c); break
except: pass
print(f'Benchmarking: {MODELS}')

## 5 · Benchmark Tasks

In [ ]:
TASKS = [
    {'name': 'short_answer', 'prompt': 'Capital of Japan? One sentence.', 'difficulty': 'easy'},
    {'name': 'summarization', 'prompt': 'Summarize in 2 sentences: ML is a subset of AI that learns from data. Includes supervised, unsupervised, RL. Used in healthcare, finance, autonomous vehicles.', 'difficulty': 'medium'},
    {'name': 'reasoning', 'prompt': 'Fields A=200kg, B=350kg, C=150kg. Which 2 fields ship >= 500kg?', 'difficulty': 'medium'},
    {'name': 'extraction', 'prompt': 'Extract JSON: Sarah Chen, 28, data scientist at Google, Mountain View.', 'difficulty': 'hard'},
]
print(f'{len(TASKS)} tasks')

## 6 · Run Benchmarks

In [ ]:
results = []
for model_name in MODELS:
    print(f'\nBenchmarking: {model_name}')
    m = ChatOllama(model=model_name, temperature=0.1)
    prompt = ChatPromptTemplate.from_messages([('human', '{prompt}')])
    chain = prompt | m | StrOutputParser()
    try: chain.invoke({'prompt': 'Hi'})  # warmup
    except: pass
    for task in TASKS:
        t0 = time.time()
        try:
            out = chain.invoke({'prompt': task['prompt']})
            lat = time.time() - t0
            wc = len(out.split())
        except Exception as e:
            out, lat, wc = '', time.time()-t0, 0
        results.append({'model': model_name, 'task': task['name'], 'latency_s': round(lat,2),
                        'word_count': wc, 'wps': round(wc/lat,1) if lat > 0 else 0})
        print(f'  {task["name"]}: {lat:.1f}s, {wc} words')
print(f'\n{len(results)} benchmarks done')

## 7 · Analyze

In [ ]:
df = pd.DataFrame(results)
print('LATENCY BY MODEL AND TASK:')
print(df.pivot_table('latency_s', 'task', 'model', 'mean').round(2).to_string())
print('\nAVERAGE BY MODEL:')
print(df.groupby('model')[['latency_s','word_count','wps']].mean().round(2).to_string())

## 8 · Save

In [ ]:
df.to_csv('benchmark_results.csv', index=False)
print('Saved.')

## 9 · Key Takeaways

- Smaller models are faster but may sacrifice quality
- Task complexity affects the quality gap more than latency
- Warmup calls matter — first inference is always slower
- Local benchmarking helps you pick the right model per task